# Comprehensive Fraud Detection System
## Model Training, Imbalance Handling, Explainability & API Development

This notebook covers:
- Fake transaction data generation
- Data preprocessing and EDA
- Class imbalance handling with SMOTE
- Training multiple models (Random Forest & XGBoost)
- Model performance comparison
- SHAP explainability analysis
- FastAPI deployment preparation

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                           roc_curve, auc, precision_recall_curve, f1_score,
                           precision_score, recall_score, accuracy_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully!")

## Section 1: Fake Transaction Data Generator

Generate synthetic transaction dataset with realistic patterns for testing and model development.

In [ ]:
class FakeTransactionGenerator:
    """Generate synthetic transaction data with realistic patterns."""

    def __init__(self, seed=42):
        """Initialize the generator with optional random seed for reproducibility."""
        np.random.seed(seed)
        
        self.products = ['W', 'H', 'S', 'C', 'R']
        self.device_types = ['desktop', 'mobile', 'tablet']
        self.os_types = ['Windows', 'MacOS', 'Android', 'iOS', 'Linux']
        self.browsers = ['Chrome', 'Safari', 'Firefox', 'Edge', 'Opera']
        self.countries = ['US', 'GB', 'CA', 'AU', 'DE', 'FR', 'JP', 'IN', 'BR', 'MX']
        self.merchant_names = ['Amazon', 'Walmart', 'Target', 'Best Buy', 'Apple',
                              'Starbucks', 'McDonald', 'Uber', 'Lyft', 'Netflix']

    def generate_transactions(self, n_samples=10000, fraud_ratio=0.1):
        """Generate synthetic transaction dataset."""
        n_fraud = int(n_samples * fraud_ratio)
        n_legitimate = n_samples - n_fraud

        # Generate legitimate transactions
        legitimate_txns = self._generate_legitimate_transactions(n_legitimate)
        
        # Generate fraudulent transactions
        fraudulent_txns = self._generate_fraudulent_transactions(n_fraud)
        
        # Combine and shuffle
        df = pd.concat([legitimate_txns, fraudulent_txns], ignore_index=True)
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        
        return df

    def _generate_legitimate_transactions(self, n):
        """Generate legitimate transaction records."""
        data = {
            'TransactionID': [f'TXN_{i:08d}' for i in range(n)],
            'TransactionAmt': np.random.lognormal(3.5, 1.5, n),
            'ProductCD': np.random.choice(self.products, n, p=[0.3, 0.25, 0.25, 0.15, 0.05]),
            'isFraud': np.zeros(n, dtype=int),
            'DayOfWeek': np.random.randint(0, 7, n),
            'Hour': np.random.randint(0, 24, n),
            'CardType': np.random.choice(['credit', 'debit'], n, p=[0.6, 0.4]),
            'DeviceType': np.random.choice(self.device_types, n),
            'OS': np.random.choice(self.os_types, n),
            'Browser': np.random.choice(self.browsers, n),
            'Country': np.random.choice(self.countries, n),
            'Merchant': np.random.choice(self.merchant_names, n),
            'Distance_km': np.random.exponential(500, n),
            'DaysSincePreviousTxn': np.random.exponential(10, n),
            'NumPreviousTxns': np.random.poisson(20, n),
        }
        
        df = pd.DataFrame(data)
        df['TransactionAmt'] = df['TransactionAmt'].round(2)
        df['Distance_km'] = df['Distance_km'].round(2)
        df['DaysSincePreviousTxn'] = df['DaysSincePreviousTxn'].round(2)
        
        return df

    def _generate_fraudulent_transactions(self, n):
        """Generate fraudulent transaction records with suspicious patterns."""
        data = {
            'TransactionID': [f'TXN_FRAUD_{i:08d}' for i in range(n)],
            'TransactionAmt': np.random.choice([np.random.uniform(1000, 5000) for _ in range(n)], n),
            'ProductCD': np.random.choice(self.products, n, p=[0.5, 0.2, 0.15, 0.1, 0.05]),
            'isFraud': np.ones(n, dtype=int),
            'DayOfWeek': np.random.randint(0, 7, n),
            'Hour': np.random.choice(range(0, 24), n),
            'CardType': np.random.choice(['credit', 'debit'], n, p=[0.8, 0.2]),
            'DeviceType': np.random.choice(self.device_types, n, p=[0.2, 0.7, 0.1]),
            'OS': np.random.choice(self.os_types, n),
            'Browser': np.random.choice(self.browsers, n),
            'Country': np.random.choice(self.countries, n),
            'Merchant': np.random.choice(self.merchant_names, n),
            'Distance_km': np.random.exponential(2000, n),
            'DaysSincePreviousTxn': np.random.exponential(2, n),
            'NumPreviousTxns': np.random.poisson(5, n),
        }
        
        df = pd.DataFrame(data)
        df['TransactionAmt'] = df['TransactionAmt'].round(2)
        df['Distance_km'] = df['Distance_km'].round(2)
        df['DaysSincePreviousTxn'] = df['DaysSincePreviousTxn'].round(2)
        
        return df

# Generate synthetic data
print("Generating synthetic transaction data...")
generator = FakeTransactionGenerator(seed=42)
df = generator.generate_transactions(n_samples=15000, fraud_ratio=0.10)

print(f"✓ Generated {len(df):,} transactions")
print(f"✓ Fraud ratio: {df['isFraud'].mean():.2%}")
print(f"✓ Dataset shape: {df.shape}")
print("\nFirst few records:")
print(df.head())

## Section 2: Data Exploration and Preprocessing

Analyze the dataset and prepare it for model training.

In [ ]:
# Exploratory Data Analysis
print("="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

print("\nDataset Info:")
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")

print("\n\nFraud Distribution:")
fraud_dist = df['isFraud'].value_counts()
print(fraud_dist)
print(f"\nFraud Rate: {df['isFraud'].mean():.2%}")

print("\n\nNumerical Features Statistics:")
print(df.describe())

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class distribution
df['isFraud'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Fraud vs Legitimate Transactions', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Transaction Type')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)

# Class distribution percentage
df['isFraud'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                                colors=['green', 'red'], labels=['Legitimate', 'Fraud'])
axes[1].set_title('Class Distribution (%)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ EDA completed!")

In [ ]:
# Data Preprocessing
print("\n" + "="*60)
print("DATA PREPROCESSING")
print("="*60)

# Create a copy for preprocessing
df_processed = df.copy()

# Drop ID columns
df_processed = df_processed.drop(['TransactionID'], axis=1)

# Identify categorical and numerical columns
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('isFraud')  # Remove target variable

print(f"\nCategorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# Encode categorical variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f"✓ Encoded {col}")

print(f"\n✓ All categorical variables encoded!")

# Feature distribution comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols[:6]):
    legitimate = df_processed[df_processed['isFraud'] == 0][col]
    fraudulent = df_processed[df_processed['isFraud'] == 1][col]
    
    axes[idx].hist(legitimate, bins=30, alpha=0.6, label='Legitimate', color='green')
    axes[idx].hist(fraudulent, bins=30, alpha=0.6, label='Fraudulent', color='red')
    axes[idx].set_title(f'{col} Distribution', fontweight='bold')
    axes[idx].legend()
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("\n✓ Data preprocessing completed!")

## Section 3: Imbalanced Data Handling with SMOTE

Handle class imbalance using SMOTE (Synthetic Minority Over-sampling Technique).

In [ ]:
print("\n" + "="*60)
print("HANDLING CLASS IMBALANCE WITH SMOTE")
print("="*60)

# Separate features and target
X = df_processed.drop('isFraud', axis=1)
y = df_processed['isFraud']

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {len(X_train):,}")
print(f"Testing set size: {len(X_test):,}")

print(f"\nBefore SMOTE:")
print(f"  Legitimate (Train): {(y_train == 0).sum():,}")
print(f"  Fraudulent (Train): {(y_train == 1).sum():,}")
print(f"  Fraud ratio (Train): {y_train.mean():.2%}")

# Apply SMOTE only to training data
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Legitimate (Train): {(y_train_smote == 0).sum():,}")
print(f"  Fraudulent (Train): {(y_train_smote == 1).sum():,}")
print(f"  Fraud ratio (Train): {y_train_smote.mean():.2%}")

print(f"\nTest set (unchanged):")
print(f"  Legitimate (Test): {(y_test == 0).sum():,}")
print(f"  Fraudulent (Test): {(y_test == 1).sum():,}")
print(f"  Fraud ratio (Test): {y_test.mean():.2%}")

# Visualize SMOTE effect
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Before SMOTE
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Before SMOTE (Training Data)', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Legitimate', 'Fraudulent'], rotation=0)

# After SMOTE
y_train_smote.value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red'])
axes[1].set_title('After SMOTE (Training Data)', fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Legitimate', 'Fraudulent'], rotation=0)

# Test set (for reference)
y_test.value_counts().plot(kind='bar', ax=axes[2], color=['green', 'red'])
axes[2].set_title('Test Set (Unchanged)', fontweight='bold')
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Count')
axes[2].set_xticklabels(['Legitimate', 'Fraudulent'], rotation=0)

plt.tight_layout()
plt.show()

print("\n✓ SMOTE applied successfully!")

In [ ]:
# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("✓ Features scaled using StandardScaler!")

## Section 4: Train Random Forest Model

Train a Random Forest classifier on the balanced training data.

In [ ]:
print("\n" + "="*60)
print("TRAINING RANDOM FOREST MODEL")
print("="*60)

# Train Random Forest classifier
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

print("\nTraining Random Forest...")
rf_model.fit(X_train_scaled, y_train_smote)
print("✓ Random Forest training completed!")

# Make predictions
rf_pred_train = rf_model.predict(X_train_scaled)
rf_pred_proba_train = rf_model.predict_proba(X_train_scaled)[:, 1]

rf_pred_test = rf_model.predict(X_test_scaled)
rf_pred_proba_test = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate on training set
print("\n--- Random Forest Training Set Performance ---")
print(f"Accuracy: {accuracy_score(y_train_smote, rf_pred_train):.4f}")
print(f"Precision: {precision_score(y_train_smote, rf_pred_train):.4f}")
print(f"Recall: {recall_score(y_train_smote, rf_pred_train):.4f}")
print(f"F1-Score: {f1_score(y_train_smote, rf_pred_train):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_train_smote, rf_pred_proba_train):.4f}")

# Evaluate on test set
print("\n--- Random Forest Test Set Performance ---")
print(f"Accuracy: {accuracy_score(y_test, rf_pred_test):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred_test):.4f}")
print(f"Recall: {recall_score(y_test, rf_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, rf_pred_test):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, rf_pred_proba_test):.4f}")

print("\n✓ Random Forest model evaluation completed!")

## Section 5: Train Alternative Model (XGBoost)

Train XGBoost classifier for comparison.

In [ ]:
print("\n" + "="*60)
print("TRAINING XGBOOST MODEL")
print("="*60)

# Calculate scale_pos_weight for imbalanced data
scale_pos_weight = (y_train_smote == 0).sum() / (y_train_smote == 1).sum()

# Train XGBoost classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr'
)

print("\nTraining XGBoost...")
xgb_model.fit(X_train_scaled, y_train_smote)
print("✓ XGBoost training completed!")

# Make predictions
xgb_pred_train = xgb_model.predict(X_train_scaled)
xgb_pred_proba_train = xgb_model.predict_proba(X_train_scaled)[:, 1]

xgb_pred_test = xgb_model.predict(X_test_scaled)
xgb_pred_proba_test = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate on training set
print("\n--- XGBoost Training Set Performance ---")
print(f"Accuracy: {accuracy_score(y_train_smote, xgb_pred_train):.4f}")
print(f"Precision: {precision_score(y_train_smote, xgb_pred_train):.4f}")
print(f"Recall: {recall_score(y_train_smote, xgb_pred_train):.4f}")
print(f"F1-Score: {f1_score(y_train_smote, xgb_pred_train):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_train_smote, xgb_pred_proba_train):.4f}")

# Evaluate on test set
print("\n--- XGBoost Test Set Performance ---")
print(f"Accuracy: {accuracy_score(y_test, xgb_pred_test):.4f}")
print(f"Precision: {precision_score(y_test, xgb_pred_test):.4f}")
print(f"Recall: {recall_score(y_test, xgb_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, xgb_pred_test):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_pred_proba_test):.4f}")

print("\n✓ XGBoost model evaluation completed!")

## Section 6: Compare Model Performance

Compare Random Forest and XGBoost models across multiple metrics.

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

# Create comparison metrics
metrics_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Random Forest': [
        accuracy_score(y_test, rf_pred_test),
        precision_score(y_test, rf_pred_test),
        recall_score(y_test, rf_pred_test),
        f1_score(y_test, rf_pred_test),
        roc_auc_score(y_test, rf_pred_proba_test)
    ],
    'XGBoost': [
        accuracy_score(y_test, xgb_pred_test),
        precision_score(y_test, xgb_pred_test),
        recall_score(y_test, xgb_pred_test),
        f1_score(y_test, xgb_pred_test),
        roc_auc_score(y_test, xgb_pred_proba_test)
    ]
})

print("\nMetrics Comparison (Test Set):")
print(metrics_comparison.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

for idx, metric in enumerate(metrics):
    values = metrics_comparison[metrics_comparison['Metric'] == metric][['Random Forest', 'XGBoost']].values.flatten()
    bars = axes[idx].bar(['Random Forest', 'XGBoost'], values, color=['#3498db', '#e74c3c'])
    axes[idx].set_title(f'{metric}', fontsize=12, fontweight='bold')
    axes[idx].set_ylim([0, 1])
    axes[idx].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.3f}', ha='center', va='bottom', fontweight='bold')

# Remove the extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

print("\n✓ Model comparison completed!")

In [ ]:
# ROC Curve Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_pred_proba_test)
roc_auc_rf = roc_auc_score(y_test, rf_pred_proba_test)

axes[0].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_rf:.3f})', color='#3498db', lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
axes[0].set_xlabel('False Positive Rate', fontweight='bold')
axes[0].set_ylabel('True Positive Rate', fontweight='bold')
axes[0].set_title('Random Forest - ROC Curve', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# XGBoost ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_pred_proba_test)
roc_auc_xgb = roc_auc_score(y_test, xgb_pred_proba_test)

axes[1].plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {roc_auc_xgb:.3f})', color='#e74c3c', lw=2)
axes[1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate', fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontweight='bold')
axes[1].set_title('XGBoost - ROC Curve', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Random Forest Confusion Matrix
rf_cm = confusion_matrix(y_test, rf_pred_test)
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Legitimate', 'Fraudulent'], yticklabels=['Legitimate', 'Fraudulent'])
axes[0].set_title('Random Forest - Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# XGBoost Confusion Matrix
xgb_cm = confusion_matrix(y_test, xgb_pred_test)
sns.heatmap(xgb_cm, annot=True, fmt='d', cmap='Reds', ax=axes[1], cbar=False,
            xticklabels=['Legitimate', 'Fraudulent'], yticklabels=['Legitimate', 'Fraudulent'])
axes[1].set_title('XGBoost - Confusion Matrix', fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("\n✓ Confusion matrices visualized!")

## Section 7: Model Explainability with SHAP

Generate SHAP values for model predictions and explain feature importance.

In [ ]:
print("\n" + "="*60)
print("SHAP EXPLAINABILITY ANALYSIS")
print("="*60)

# Use a sample of test data for SHAP analysis (to avoid long computation)
X_test_sample = X_test_scaled.iloc[:500]  # Use 500 samples for faster computation

print("\nGenerating SHAP explanations for XGBoost model...")

# Create SHAP explainer
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_sample)

print("✓ SHAP values computed!")

# Global Feature Importance - Summary Plot
print("\n--- SHAP Summary Plot (Mean Absolute SHAP values) ---")
shap.summary_plot(shap_values, X_test_sample, plot_type="bar", show=True)

print("\n--- SHAP Summary Plot (All points) ---")
shap.summary_plot(shap_values, X_test_sample, show=True)

print("\n✓ SHAP plots generated!")

In [ ]:
# Analyze individual predictions
print("\n--- Individual Prediction Explanations ---")

# Find a fraudulent prediction
fraud_indices = np.where(y_test.iloc[:500].values == 1)[0]

if len(fraud_indices) > 0:
    fraud_idx = fraud_indices[0]
    print(f"\nExplaining prediction for fraud case {fraud_idx}:")
    print(f"True Label: Fraudulent (1)")
    print(f"Predicted Probability: {xgb_pred_proba_test[fraud_idx]:.4f}")
    
    # Force plot for individual prediction
    shap.force_plot(explainer.expected_value, shap_values[fraud_idx], X_test_sample.iloc[fraud_idx], show=True)

# Find a legitimate prediction
legit_indices = np.where(y_test.iloc[:500].values == 0)[0]

if len(legit_indices) > 0:
    legit_idx = legit_indices[0]
    print(f"\nExplaining prediction for legitimate case {legit_idx}:")
    print(f"True Label: Legitimate (0)")
    print(f"Predicted Probability: {xgb_pred_proba_test[legit_idx]:.4f}")
    
    # Force plot for individual prediction
    shap.force_plot(explainer.expected_value, shap_values[legit_idx], X_test_sample.iloc[legit_idx], show=True)

print("\n✓ Individual prediction explanations completed!")

In [ ]:
# Feature importance from SHAP
feature_importance_shap = pd.DataFrame({
    'Feature': X_test_sample.columns,
    'SHAP_Importance': np.abs(shap_values).mean(axis=0)
}).sort_values('SHAP_Importance', ascending=False)

print("\nTop 10 Most Important Features (by SHAP):")
print(feature_importance_shap.head(10).to_string(index=False))

# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# SHAP Feature Importance
axes[0].barh(feature_importance_shap['Feature'].head(15), feature_importance_shap['SHAP_Importance'].head(15), color='#3498db')
axes[0].set_xlabel('Mean |SHAP value|', fontweight='bold')
axes[0].set_title('Feature Importance (SHAP)', fontweight='bold')
axes[0].invert_yaxis()

# XGBoost Feature Importance
xgb_importance = pd.DataFrame({
    'Feature': X_test_sample.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

axes[1].barh(xgb_importance['Feature'].head(15), xgb_importance['Importance'].head(15), color='#e74c3c')
axes[1].set_xlabel('Importance', fontweight='bold')
axes[1].set_title('Feature Importance (XGBoost)', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n✓ Feature importance visualization completed!")

## Section 8: Build FastAPI Prediction API

Prepare the best model and create code for FastAPI deployment.

In [ ]:
print("\n" + "="*60)
print("MODEL DEPLOYMENT PREPARATION")
print("="*60)

# Select the best model based on ROC-AUC score
print(f"\nRandom Forest ROC-AUC: {roc_auc_score(y_test, rf_pred_proba_test):.4f}")
print(f"XGBoost ROC-AUC: {roc_auc_score(y_test, xgb_pred_proba_test):.4f}")

best_model = xgb_model if roc_auc_score(y_test, xgb_pred_proba_test) > roc_auc_score(y_test, rf_pred_proba_test) else rf_model
best_model_name = "XGBoost" if best_model == xgb_model else "Random Forest"

print(f"\n✓ Best Model: {best_model_name}")

# Save models and preprocessing objects
import pickle
import os

model_dir = "/Users/mansidaksingh/capstone_project/Fraud-Detection-System/model"
os.makedirs(model_dir, exist_ok=True)

# Save best model
with open(f'{model_dir}/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save preprocessing objects
with open(f'{model_dir}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open(f'{model_dir}/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# Save feature names
with open(f'{model_dir}/feature_names.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

print(f"\n✓ Models saved to {model_dir}/")
print(f"  - best_model.pkl")
print(f"  - scaler.pkl")
print(f"  - label_encoders.pkl")
print(f"  - feature_names.pkl")

In [ ]:
# Generate FastAPI code template
fastapi_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import pickle
import numpy as np
import pandas as pd
from typing import List, Optional
import shap

app = FastAPI(title="Fraud Detection API", version="1.0.0")

# Load models and preprocessing objects
with open("model/best_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("model/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("model/label_encoders.pkl", "rb") as f:
    label_encoders = pickle.load(f)

with open("model/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

# Load SHAP explainer
explainer = shap.TreeExplainer(model)

# Define request models
class Transaction(BaseModel):
    """Single transaction for prediction"""
    TransactionAmt: float = Field(..., gt=0, description="Transaction amount")
    ProductCD: str = Field(..., description="Product code")
    DayOfWeek: int = Field(..., ge=0, le=6, description="Day of week")
    Hour: int = Field(..., ge=0, le=23, description="Hour of day")
    CardType: str = Field(..., description="Card type (credit/debit)")
    DeviceType: str = Field(..., description="Device type")
    OS: str = Field(..., description="Operating system")
    Browser: str = Field(..., description="Browser type")
    Country: str = Field(..., description="Country code")
    Merchant: str = Field(..., description="Merchant name")
    Distance_km: float = Field(..., ge=0, description="Distance in km")
    DaysSincePreviousTxn: float = Field(..., ge=0, description="Days since previous transaction")
    NumPreviousTxns: int = Field(..., ge=0, description="Number of previous transactions")

class PredictionResponse(BaseModel):
    """API response for prediction"""
    prediction: int = Field(..., description="0: Legitimate, 1: Fraudulent")
    fraud_probability: float = Field(..., description="Probability of fraud")
    confidence: float = Field(..., description="Confidence score")
    top_features: List[dict] = Field(..., description="Top contributing features")

class BatchPredictionRequest(BaseModel):
    """Batch prediction request"""
    transactions: List[Transaction] = Field(..., description="List of transactions")

@app.get("/health")
async def health_check():
    """Health check endpoint"""
    return {"status": "healthy", "model": "Fraud Detection API"}

@app.post("/predict", response_model=PredictionResponse)
async def predict_fraud(transaction: Transaction):
    """
    Predict if a single transaction is fraudulent.
    
    Returns:
        - prediction: 0 (Legitimate) or 1 (Fraudulent)
        - fraud_probability: Probability score for fraud
        - confidence: Confidence in prediction
        - top_features: Top 5 features contributing to prediction
    """
    try:
        # Prepare features
        features_dict = transaction.dict()
        df_input = pd.DataFrame([features_dict])
        
        # Encode categorical variables
        for col, encoder in label_encoders.items():
            if col in df_input.columns:
                df_input[col] = df_input[col].map(
                    lambda x: encoder.transform([str(x)])[0] if str(x) in encoder.classes_ else -1
                )
        
        # Ensure all features are present
        df_input = df_input.reindex(columns=feature_names, fill_value=0)
        
        # Scale features
        df_scaled = scaler.transform(df_input)
        
        # Make prediction
        prediction = model.predict(df_scaled)[0]
        fraud_probability = model.predict_proba(df_scaled)[0][1]
        confidence = max(model.predict_proba(df_scaled)[0])
        
        # Generate SHAP explanation
        shap_values = explainer.shap_values(df_scaled)
        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'shap_value': np.abs(shap_values[0])
        }).sort_values('shap_value', ascending=False).head(5)
        
        top_features = [
            {"feature": row['feature'], "importance": float(row['shap_value'])}
            for _, row in feature_importance.iterrows()
        ]
        
        return PredictionResponse(
            prediction=int(prediction),
            fraud_probability=float(fraud_probability),
            confidence=float(confidence),
            top_features=top_features
        )
    
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/predict_batch")
async def predict_batch(request: BatchPredictionRequest):
    """
    Predict fraud for multiple transactions.
    
    Returns:
        List of predictions with probabilities
    """
    try:
        results = []
        for transaction in request.transactions:
            result = await predict_fraud(transaction)
            results.append(result)
        
        return {
            "total_transactions": len(results),
            "predictions": results,
            "fraud_count": sum(1 for r in results if r.prediction == 1)
        }
    
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.get("/model_info")
async def model_info():
    """Get information about the model"""
    return {
        "model_type": type(model).__name__,
        "features": feature_names,
        "total_features": len(feature_names),
        "version": "1.0.0"
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print("FastAPI Application Code Template:")
print("="*60)
print(fastapi_code)

print("\n✓ FastAPI code template generated!")

## Summary

This comprehensive notebook covers all aspects of the fraud detection system:

### ✅ Completed Tasks:

1. **Data Generation**: Created a realistic synthetic dataset with balanced fraud scenarios
2. **Imbalance Handling**: Applied SMOTE to handle class imbalance effectively
3. **Model Training**: Trained both Random Forest and XGBoost classifiers
4. **Model Comparison**: Evaluated performance using multiple metrics (ROC-AUC, Precision, Recall, F1-Score)
5. **Explainability**: Generated SHAP values for model interpretability
6. **API Preparation**: Created FastAPI code template for deployment

### 📊 Key Results:
- Successfully balanced imbalanced dataset using SMOTE
- Compared two powerful classifiers with comprehensive evaluation metrics
- Generated actionable SHAP explanations for individual predictions
- Prepared production-ready FastAPI application code

### 🚀 Next Steps:
1. Deploy the FastAPI application to production
2. Set up monitoring and alerting for predictions
3. Continuously evaluate model performance on new data
4. Retrain the model periodically with updated data